In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("lakehouse").getOrCreate()



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 16:26:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Grab your existing active notebook Spark Session
spark = SparkSession.builder.getOrCreate()

# 2. Path to two files directly from your bronze bucket
test_paths = [
    "s3a://lakehouse/bronze/ohlcv/1min/AAPL_raw_1min.parquet",
    "s3a://lakehouse/bronze/ohlcv/1min/ABNB_raw_1min.parquet"
]

print("--- Reading raw parquet test files ---")
# Explicitly selecting the hidden column '_metadata' alongside everything else
df_raw = spark.read.parquet(*test_paths).select("*", "_metadata")

# 3. Apply the bulletproof hidden metadata extraction regex
df_with_ticker = df_raw.withColumn(
    "ticker", 
    F.regexp_extract(F.col("_metadata.file_name"), r"([^/]+)_raw", 1)
)

print("\n--- [TEST 1] Checking Ticker Extraction Before Shuffle ---")
# Changed "date" to "datetime" to match your schema
df_with_ticker.select("datetime", "Open", "Close", "ticker").show(5)

# 4. Stress Test: Trigger a heavy window partition
# Changed "date" to "datetime" here as well
window_spec = Window.partitionBy("ticker", "datetime").orderBy(F.lit(1))
df_shuffled = df_with_ticker.withColumn("_row_num", F.row_number().over(window_spec)) \
                            .filter(F.col("_row_num") == 1) \
                            .drop("_row_num")

print("\n--- [TEST 2] Checking Ticker Columns After Shuffle Boundary ---")
df_shuffled.select("datetime", "Open", "Close", "ticker").show(5)

--- Reading raw parquet test files ---

--- [TEST 1] Checking Ticker Extraction Before Shuffle ---
+-------------------+----+-----+------+
|           datetime|Open|Close|ticker|
+-------------------+----+-----+------+
|1041240600000000000|0.23| 0.23|  AAPL|
|1041240660000000000|0.23| 0.23|  AAPL|
|1041240720000000000|0.23| 0.23|  AAPL|
|1041240780000000000|0.23| 0.23|  AAPL|
|1041240840000000000|0.23| 0.23|  AAPL|
+-------------------+----+-----+------+
only showing top 5 rows


--- [TEST 2] Checking Ticker Columns After Shuffle Boundary ---


[Stage 11:==================================================>     (10 + 1) / 11]

+-------------------+----+-----+------+
|           datetime|Open|Close|ticker|
+-------------------+----+-----+------+
|1041241140000000000|0.23| 0.23|  AAPL|
|1041241500000000000|0.23| 0.23|  AAPL|
|1041241740000000000|0.23| 0.23|  AAPL|
|1041242280000000000|0.23| 0.23|  AAPL|
|1041242340000000000|0.23| 0.23|  AAPL|
+-------------------+----+-----+------+
only showing top 5 rows



In [7]:
df  =spark.read.parquet("s3a://lakehouse/bronze/ohlcv/1min/AAPL_raw_1min.parquet")

In [8]:
df.show()


+-------------------+----+----+----+-----+-------+---------+
|           datetime|Open|High| Low|Close| Volume|   source|
+-------------------+----+----+----+-----+-------+---------+
|1041240600000000000|0.23|0.23|0.23| 0.23|2285864|pitrading|
|1041240660000000000|0.23|0.23|0.23| 0.23| 683200|pitrading|
|1041240720000000000|0.23|0.23|0.23| 0.23| 441280|pitrading|
|1041240780000000000|0.23|0.23|0.23| 0.23| 112000|pitrading|
|1041240840000000000|0.23|0.23|0.23| 0.23| 495600|pitrading|
|1041240900000000000|0.23|0.23|0.23| 0.23| 196000|pitrading|
|1041240960000000000|0.23|0.23|0.23| 0.23| 246456|pitrading|
|1041241020000000000|0.23|0.23|0.23| 0.23| 112000|pitrading|
|1041241080000000000|0.23|0.23|0.23| 0.23| 686952|pitrading|
|1041241140000000000|0.23|0.23|0.23| 0.23| 380800|pitrading|
|1041241200000000000|0.23|0.23|0.23| 0.23| 285600|pitrading|
|1041241260000000000|0.23|0.23|0.23| 0.23| 751520|pitrading|
|1041241320000000000|0.23|0.23|0.23| 0.23| 414400|pitrading|
|1041241380000000000|0.2

In [4]:
bronze = spark.read.option('header',True).option('inferSchema',True).parquet("s3a://lakehouse/silver/ohlcv_35d4ba35-a8e3-4a77-b77d-cc2f0c071be2/data")

In [5]:
bronze.count()

1234511210

In [6]:
bronze.filter(bro

+-------------------+-----+-----+-----+-----+------+---------+------+
|           datetime| Open| High|  Low|Close|Volume|   source|ticker|
+-------------------+-----+-----+-----+-----+------+---------+------+
|1041240660000000000|20.93|20.96|20.93|20.96| 10800|pitrading|      |
|1041240720000000000|20.96|20.96|20.96|20.96|   300|pitrading|      |
|1041240780000000000|20.93|20.93|20.93|20.93|   600|pitrading|      |
|1041240840000000000|20.97|20.97|20.97|20.97|   100|pitrading|      |
|1041240900000000000|20.93|20.93|20.93|20.93|   200|pitrading|      |
|1041241020000000000|20.94|20.94|20.94|20.94|   600|pitrading|      |
|1041241080000000000|20.98|20.98|20.98|20.98|  1200|pitrading|      |
|1041241140000000000|20.98|20.98|20.96|20.96|  1900|pitrading|      |
|1041241260000000000|20.96|20.96|20.96|20.96|  2200|pitrading|      |
|1041241320000000000|20.96|20.96|20.93|20.93|   300|pitrading|      |
|1041241380000000000|20.93|20.93|20.93|20.93|   300|pitrading|      |
|1041241440000000000